In [6]:

from pathlib import Path
import json
import pandas as pd
import numpy as np


# ==================================================
# 1. PROJECT PATHS
# ==================================================

CURRENT_DIR = Path.cwd()

# Find project root from the current working directory
PROJECT_ROOT = CURRENT_DIR

while (
    PROJECT_ROOT.name != "hitl-tabular-data-preparation"
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "hitl-tabular-data-preparation":
    raise FileNotFoundError(
        f"Project root not found. Current directory: {CURRENT_DIR}"
    )

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "corrupted"
    / "seed_42"
    / "corrupted_train.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "results" / "operational"

OUTPUT_JSON = OUTPUT_DIR / "column_profiles_seed_42.json"
OUTPUT_CSV = OUTPUT_DIR / "column_profiles_seed_42.csv"


# ==================================================
# 2. COLUMN DEFINITIONS
# ==================================================

NUMERIC_COLUMNS = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous",
]

TARGET_COLUMN = "y"

MAX_TOP_VALUES = 20
MAX_SAMPLE_VALUES = 10


# ==================================================
# 3. HELPER FUNCTIONS
# ==================================================

def safe_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    return str(value)


def get_top_values(series, limit=MAX_TOP_VALUES):
    counts = series.value_counts(dropna=False).head(limit)

    return [
        {
            "value": safe_value(value),
            "count": int(count),
            "percentage": round(
                count / len(series) * 100, 2
            )
        }
        for value, count in counts.items()
    ]


# ==================================================
# 4. NUMERIC COLUMN PROFILE
# ==================================================

def profile_numeric_column(series):

    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()

    profile = {
        "column_type": "numeric",
        "missing_count": int(series.isna().sum()),
        "missing_percentage": round(
            series.isna().mean() * 100, 2
        ),
        "unique_count": int(series.nunique(dropna=True)),
        "valid_numeric_count": int(len(valid)),
    }

    if len(valid) == 0:
        profile["statistics"] = None
        profile["iqr_candidates"] = None
        return profile

    q1 = valid.quantile(0.25)
    median = valid.median()
    q3 = valid.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    candidates = valid[
        (valid < lower_bound) | (valid > upper_bound)
    ]

    profile["statistics"] = {
        "min": safe_value(valid.min()),
        "max": safe_value(valid.max()),
        "mean": round(float(valid.mean()), 3),
        "median": round(float(median), 3),
        "standard_deviation": round(float(valid.std()), 3),
        "q1": round(float(q1), 3),
        "q3": round(float(q3), 3),
    }

    profile["iqr_candidates"] = {
        "lower_bound": round(float(lower_bound), 3),
        "upper_bound": round(float(upper_bound), 3),
        "count": int(len(candidates)),
        "percentage": round(
            len(candidates) / len(valid) * 100, 2
        ),
        "sample_values": [
            safe_value(v)
            for v in candidates.head(MAX_SAMPLE_VALUES)
        ],
    }

    return profile


# ==================================================
# 5. CATEGORICAL COLUMN PROFILE
# ==================================================

def profile_categorical_column(series):

    non_missing = series.dropna().astype(str)

    whitespace_count = int(
        (non_missing != non_missing.str.strip()).sum()
    )

    case_variation_count = int(
        (non_missing != non_missing.str.lower()).sum()
    )

    rare_values = (
        non_missing.value_counts()
        [lambda counts: counts <= 5]
        .head(MAX_SAMPLE_VALUES)
    )

    return {
        "column_type": "categorical",
        "missing_count": int(series.isna().sum()),
        "missing_percentage": round(
            series.isna().mean() * 100, 2
        ),
        "unique_count": int(series.nunique(dropna=True)),
        "whitespace_variation_count": whitespace_count,
        "case_variation_count": case_variation_count,
        "top_values": get_top_values(series),
        "rare_values": [
            {
                "value": safe_value(value),
                "count": int(count),
            }
            for value, count in rare_values.items()
        ],
        "sample_values": [
            safe_value(v)
            for v in series.dropna()
            .drop_duplicates()
            .head(MAX_SAMPLE_VALUES)
        ],
    }


# ==================================================
# 6. BUILD ALL COLUMN PROFILES
# ==================================================

def build_profiles(df):

    profiles = {}

    for column in df.columns:

        if column == TARGET_COLUMN:
            continue

        series = df[column]

        if column in NUMERIC_COLUMNS:
            profile = profile_numeric_column(series)
        else:
            profile = profile_categorical_column(series)

        profiles[column] = profile

    return profiles


# ==================================================
# 7. MAIN
# ==================================================

def main():

    print("=" * 60)
    print("COLUMN PROFILING")
    print("=" * 60)

    print(f"\nProject root: {PROJECT_ROOT}")
    print(f"Input file: {INPUT_FILE}")

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Input file not found: {INPUT_FILE}"
        )

    df = pd.read_csv(
        INPUT_FILE,
        keep_default_na=True
    )

    print(f"\nDataset shape: {df.shape}")

    if TARGET_COLUMN not in df.columns:
        raise ValueError("Target column 'y' is missing.")

    if "duration" in df.columns:
        raise ValueError(
            "Unexpected leakage column 'duration' found."
        )

    print("\nBuilding column profiles...")

    profiles = build_profiles(df)

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        OUTPUT_JSON,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            profiles,
            file,
            indent=4,
            ensure_ascii=False,
            allow_nan=False
        )

    overview = []

    for column, profile in profiles.items():

        overview.append({
            "column": column,
            "type": profile["column_type"],
            "missing_count": profile["missing_count"],
            "missing_percentage": profile["missing_percentage"],
            "unique_count": profile["unique_count"],
        })

    overview_df = pd.DataFrame(overview)

    overview_df.to_csv(
        OUTPUT_CSV,
        index=False
    )

    print("\n" + "=" * 60)
    print("COLUMN PROFILING COMPLETED")
    print("=" * 60)

    print(f"\nProfiles created: {len(profiles)}")
    print(f"JSON saved: {OUTPUT_JSON}")
    print(f"CSV saved: {OUTPUT_CSV}")

    print("\nColumn overview:\n")
    print(overview_df.to_string(index=False))

    return profiles, overview_df


# ==================================================
# 8. RUN
# ==================================================

profiles, overview_df = main()


COLUMN PROFILING

Project root: C:\Users\ayman\OneDrive\Desktop\hitl-tabular-data-preparation
Input file: C:\Users\ayman\OneDrive\Desktop\hitl-tabular-data-preparation\data\corrupted\seed_42\corrupted_train.csv

Dataset shape: (4000, 16)

Building column profiles...

COLUMN PROFILING COMPLETED

Profiles created: 15
JSON saved: C:\Users\ayman\OneDrive\Desktop\hitl-tabular-data-preparation\results\operational\column_profiles_seed_42.json
CSV saved: C:\Users\ayman\OneDrive\Desktop\hitl-tabular-data-preparation\results\operational\column_profiles_seed_42.csv

Column overview:

   column        type  missing_count  missing_percentage  unique_count
      age     numeric             97                2.43           103
      job categorical            109                2.72            84
  marital categorical            100                2.50            26
education categorical             79                1.98            37
  default categorical            109                2.72         